This notebook contains code related to the prediction of in vitro IL-2 autocrine signaling in CD8⁺ T cells

In [149]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

df_res = pd.read_csv("results/simulations_CD8cc.csv")
df_res["Cell_type"] = [cell.split("_")[0] for cell in df_res["Cell_type"]]
df_res.loc[df_res["Cell_type"] == "arrested", "pSTAT5_MFI"] = 549.1803278688524
df_res.loc[df_res["Cell_type"] == "non-arrested", "pSTAT5_MFI"] = 609.8360655737705
df_res.loc[df_res["Cell_type"] == "released", "pSTAT5_MFI"] = 839.3442622950819
df_res.loc[df_res["Cell_type"] == "unstimulated", "pSTAT5_MFI"] = 263.9344262295082

pcc = pearsonr(df_res["pSTAT5_MFI"], df_res["Result"])
scc = spearmanr(df_res["pSTAT5_MFI"], df_res["Result"])
print("PCC: " + str(pcc))
print("SCC: " + str(scc))

# Order (low -> high pSTAT5 experimental MFI) and style taken from the reference legend image
cell_order = ["unstimulated", "arrested", "non-arrested", "released"]
cell_style = {
    "unstimulated": {"label": "Unstimulated", "color": "#A3A3A3", "marker": "^"},
    "arrested":     {"label": "Arrested",     "color": "#4C6AA0", "marker": "D"},
    "non-arrested": {"label": "Non-arrested", "color": "#46815A", "marker": "s"},
    "released":     {"label": "Released",     "color": "#F7AA62", "marker": "o"},
}

data = [df_res.loc[df_res["Cell_type"] == ct, "Result"].dropna().to_list() for ct in cell_order]
positions = [df_res.loc[df_res["Cell_type"] == ct, "pSTAT5_MFI"].iloc[0] for ct in cell_order]

# Box width scaled to the actual numeric spread of the x-axis positions
x_range = max(positions) - min(positions)
box_width = x_range * 0.2

fig, ax = plt.subplots(1, 1, figsize=(8, 7), dpi=400)

bp = ax.boxplot(
    data,
    positions=positions,
    widths=box_width,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(color="black", linewidth=2),
    capprops=dict(color="black", linewidth=2),
    boxprops=dict(edgecolor="black", linewidth=2),
    zorder=2,
)

# Boxes get the strong/solid cell-type color; points get a lighter (white) fill
# with a colored edge, so the two layers stay visually distinct.
for patch, ct in zip(bp["boxes"], cell_order):
    patch.set_facecolor(cell_style[ct]["color"])
    patch.set_alpha(0.85)

for pos, ct in zip(positions, cell_order):
    values = df_res.loc[df_res["Cell_type"] == ct, "Result"].dropna().to_list()
    style = cell_style[ct]
    x = np.random.normal(pos, box_width * 0, size=len(values))
    ax.scatter(
        x, values,
        facecolor="white",
        edgecolor=style["color"],
        marker=style["marker"],
        linewidth=1.8,
        alpha=1.0,
        s=80,
        zorder=3,
    )



# Spine formatting
ax.spines["bottom"].set_linewidth(3.5)
ax.spines["left"].set_linewidth(3.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting - real numeric pSTAT5_MFI positions on the x-axis (as in plot 1);
# cell-type identity is in the legend
ax.xaxis.set_tick_params(width=3.5, length=8)
ax.yaxis.set_tick_params(width=3.5, length=8)
ax.tick_params(axis="x", labelsize=21)
ax.tick_params(axis="y", labelsize=21)

# Labels
ax.set_ylabel("pSTAT5 (Model, molecules per cell)", fontsize=21)
ax.set_xlabel("pSTAT5 (Experimental, gMFI)", fontsize=21)

# Legend: cell-type markers/colors (from reference image) + correlation stats
legend_handles = [
    plt.Line2D([0], [0], marker=cell_style[ct]["marker"], color="w",
               markerfacecolor=cell_style[ct]["color"], markeredgecolor="black",
               markersize=11, label=cell_style[ct]["label"])
    for ct in cell_order
]
legend_handles += [
    plt.Line2D([0], [0], color="none", label=f"PCC = {pcc[0]:.2f}"),
    plt.Line2D([0], [0], color="none", label=f"SCC = {scc[0]:.2f}"),
]

leg = ax.legend(
    handles=legend_handles,
    loc="upper left",
    fontsize=14,
    frameon=True,
    handletextpad=0.6,
    borderaxespad=0.5,
    labelspacing=0.5,
)
leg.get_frame().set_edgecolor("grey")
leg.get_frame().set_linewidth(1.2)
ax.set_xticks([200, 300, 400, 500, 600, 700, 800, 900])
ax.set_xticklabels(["200", "300", "400", "500", "600", "700", "800", "900"])
plt.savefig(
    "figures/fit_param/Plot_box_pSTAT5_CD8_cell_cycle.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

PCC: PearsonRResult(statistic=0.9429172246654275, pvalue=4.5502896974700453e-08)
SCC: SignificanceResult(statistic=0.9580157188935153, pvalue=5.516614481735588e-09)
